# Dia 10 — Dois grupos, um número 🔗

**Linux para OSINT — 21 dias** · baseado no curso de [cipher387](https://github.com/cipher387/linux-for-OSINT-21-day) (MIT)

📘 Texto completo do dia: [dia-10-utilitarios-de-texto](https://github.com/Ridd1kulusC0d3r/linux-para-osint-21-dias/tree/main/dias/dia-10-utilitarios-de-texto)

## 🎬 Missão

> Duas vítimas, sem se conhecerem, mandaram ao Núcleo a lista de membros dos
> grupos de "suporte" onde caíram: **Cashback VIP** e **Suporte Pix**. Uma
> exportou em CSV, a outra copiou de um jeito todo próprio — ponto e vírgula,
> colunas trocadas, apelidos em MAIÚSCULAS.
>
> Bento coloca as duas lado a lado na tela:
> — Golpista economiza chip. Se o mesmo número administra os dois grupos, é a
> nossa ponte.
>
> Jurema, da porta:
> — São telefones de gente. Cruza, acha o que importa e **não** espalha o resto.

**Sua tarefa:** encontrar o **único** telefone que aparece nos dois grupos e
montar a flag com o apelido dele em cada grupo, em minúsculas:
`VAZANTE{<apelido_no_grupo_A>_<apelido_no_grupo_B>}`.

## ⚙️ Preparação

Rode a célula abaixo **uma vez**. Ela baixa o repositório para dentro do Colab e define a variável `L4O_HOME`, que todas as outras células usam.

> No Colab você é `root` e o `apt` funciona. Cada célula `%%bash` roda num processo separado: variáveis e `cd` não passam de uma célula para outra.

In [ ]:
import os, subprocess
from pathlib import Path

# Se já estamos dentro do repositório (Codespaces, máquina local), usa ele.
aqui = Path.cwd()
candidatos = [aqui, *aqui.parents]
raiz = next((p for p in candidatos if (p / 'lib' / 'comum.sh').exists()), None)

if raiz is None:  # Colab: clona o repositório
    raiz = Path('/content/linux-para-osint-21-dias')
    if not raiz.exists():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Ridd1kulusC0d3r/linux-para-osint-21-dias', str(raiz)], check=True)

os.environ['L4O_HOME'] = str(raiz)
os.environ['L4O_CORES'] = '1'
print('Repositório em', raiz)

### Olhando as duas listas

```text
Duas exportações de grupos de "suporte" d'Os Vazantes. Repare: formatos
diferentes! O grupo A é CSV com vírgula; o grupo B usa ponto e vírgula,
outra ordem de colunas e apelidos em MAIÚSCULAS. Mundo real é assim.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-10-utilitarios-de-texto" && source "$L4O_HOME/lib/comum.sh"
titulo "As primeiras linhas de cada grupo"
head -n 4 desafio/grupo-a-cashback-vip.csv
echo "---"
head -n 4 desafio/grupo-b-suporte-pix.txt

### wc: medindo o tamanho

```text
`wc` (word count) conta linhas (-l), palavras (-w) e caracteres (-c / -m).
Sem opção, mostra as três contagens.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-10-utilitarios-de-texto" && source "$L4O_HOME/lib/comum.sh"
titulo "Contando"
wc desafio/grupo-a-cashback-vip.csv
wc -l desafio/*.csv desafio/*.txt

### cat: concatenar (juntar) arquivos

```text
`cat` não serve só para ler: o nome vem de "concatenate".
  cat a b > c        junta a e b num arquivo novo c
  cat *.txt > d/x    junta todos os .txt dentro de outra pasta
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-10-utilitarios-de-texto" && source "$L4O_HOME/lib/comum.sh"
titulo "Juntando arquivos"
mkdir -p treino/alltext
echo "Primeiro arquivo" > treino/file1.txt
echo "Segundo arquivo" > treino/file2.txt
cat treino/file1.txt treino/file2.txt > treino/file3.txt
cat treino/file3.txt
cat treino/*.txt > treino/alltext/file4.txt
wc -l treino/alltext/file4.txt
rm -r treino

### cut: recortando colunas e caracteres

```text
  cut -c 1        só o 1º caractere de cada linha
  cut -c 5-10     do 5º ao 10º caractere
  cut -d, -f1     separa por vírgula (-d = delimitador) e pega o campo 1 (-f)
  cut -d';' -f2   o ; precisa de aspas, senão o Bash acha que é fim de comando
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-10-utilitarios-de-texto" && source "$L4O_HOME/lib/comum.sh"
titulo "cut por caractere e por campo"
head -n 3 desafio/grupo-a-cashback-vip.csv | cut -c 1
head -n 3 desafio/grupo-a-cashback-vip.csv | cut -c 5-10
head -n 3 desafio/grupo-a-cashback-vip.csv | cut -d, -f1
head -n 3 desafio/grupo-b-suporte-pix.txt | cut -d';' -f2

### tail -n +2: pulando o cabeçalho

```text
A primeira linha de cada lista é o cabeçalho. `tail -n +2` começa da linha 2.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-10-utilitarios-de-texto" && source "$L4O_HOME/lib/comum.sh"
titulo "Sem cabeçalho"
tail -n +2 desafio/grupo-a-cashback-vip.csv | head -n 3

### sort: ordenando

```text
  sort       ordem alfabética        sort -r    ordem invertida
  sort -n    numérica                sort -u    ordena e remove repetidos
  sort -t, -k2   ordena pela coluna 2 usando vírgula como separador
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-10-utilitarios-de-texto" && source "$L4O_HOME/lib/comum.sh"
titulo "Ordenando"
tail -n +2 desafio/grupo-a-cashback-vip.csv | sort | head -n 3
echo "--- invertido"
tail -n +2 desafio/grupo-a-cashback-vip.csv | sort -r | head -n 3
echo "--- pela coluna apelido"
tail -n +2 desafio/grupo-a-cashback-vip.csv | sort -t, -k2 | head -n 3

### uniq: repetidos

```text
`uniq` só remove repetidos que estão GRUDADOS (linhas vizinhas). Por isso
quase sempre vem depois de `sort`.
  uniq       remove duplicatas vizinhas
  uniq -c    conta quantas vezes cada linha aparece
  uniq -d    mostra só as que se repetem
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-10-utilitarios-de-texto" && source "$L4O_HOME/lib/comum.sh"
titulo "Linhas duplicadas na exportação do grupo A"
tail -n +2 desafio/grupo-a-cashback-vip.csv | wc -l
tail -n +2 desafio/grupo-a-cashback-vip.csv | sort | uniq | wc -l
tail -n +2 desafio/grupo-a-cashback-vip.csv | sort | uniq -d

### O ranking: sort | uniq -c | sort -rn

```text
Combinação clássica de OSINT: "quem aparece mais?"
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-10-utilitarios-de-texto" && source "$L4O_HOME/lib/comum.sh"
titulo "Telefones que mais se repetem no grupo A"
tail -n +2 desafio/grupo-a-cashback-vip.csv | cut -d, -f1 | sort | uniq -c | sort -rn | head -n 5

### tr: trocando caracteres

```text
  tr '[:upper:]' '[:lower:]'   maiúsculas -> minúsculas
  tr ';' ','       troca um separador por outro
  tr -d '() -'     apaga parênteses, hífens e espaços
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-10-utilitarios-de-texto" && source "$L4O_HOME/lib/comum.sh"
titulo "Padronizando o grupo B"
tail -n +2 desafio/grupo-b-suporte-pix.txt | head -n 3 | tr '[:upper:]' '[:lower:]' | tr ';' ','
head -n 3 desafio/grupo-b-suporte-pix.txt | cut -d';' -f2 | tr -d '() -'

### comm: o que está em um, no outro ou nos dois

```text
`comm` compara DUAS listas JÁ ORDENADAS e imprime três colunas:
  1ª só no arquivo 1 · 2ª só no arquivo 2 · 3ª nos dois
  comm -12  esconde as colunas 1 e 2 -> sobra só o que é comum
  comm -23  só o que está no 1º e não no 2º
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-10-utilitarios-de-texto" && source "$L4O_HOME/lib/comum.sh"
titulo "Telefones de cada grupo, ordenados e sem repetição"
mkdir -p treino
tail -n +2 desafio/grupo-a-cashback-vip.csv | cut -d, -f1 | sort -u > treino/tel-a.txt
tail -n +2 desafio/grupo-b-suporte-pix.txt | cut -d';' -f2 | sort -u > treino/tel-b.txt
wc -l treino/tel-a.txt treino/tel-b.txt
echo "--- só no grupo A (primeiros 3):"
comm -23 treino/tel-a.txt treino/tel-b.txt | head -n 3
dica "Agora tente comm -12 você mesmo: é o desafio de hoje."
rm -r treino

### diff: o que mudou de ontem para hoje?

```text
`diff` mostra as diferenças linha a linha entre dois arquivos.
  <  linha que só existe no 1º (saiu)      >  linha que só existe no 2º (entrou)
  -u  formato "unificado" (o mesmo do Git): - saiu, + entrou
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-10-utilitarios-de-texto" && source "$L4O_HOME/lib/comum.sh"
titulo "Quem entrou e quem saiu do grupo A"
diff desafio/grupo-a-ontem.csv desafio/grupo-a-cashback-vip.csv || true
echo "--- formato unificado"
diff -u desafio/grupo-a-ontem.csv desafio/grupo-a-cashback-vip.csv | grep '^[+-](' || true

### paste: colando arquivos lado a lado

```text
`paste` junta arquivos COLUNA a COLUNA (o cat junta linha embaixo de linha).
  paste a b          separa por TAB
  paste -d, a b      separa por vírgula
  paste -sd+ arq     junta todas as linhas de um arquivo numa só, com +
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-10-utilitarios-de-texto" && source "$L4O_HOME/lib/comum.sh"
titulo "Lado a lado"
mkdir -p treino
tail -n +2 desafio/grupo-b-suporte-pix.txt | head -n 3 | cut -d';' -f2 > treino/tel.txt
tail -n +2 desafio/grupo-b-suporte-pix.txt | head -n 3 | cut -d';' -f1 | tr '[:upper:]' '[:lower:]' > treino/apelido.txt
paste -d, treino/tel.txt treino/apelido.txt
conta="$(printf '3\n5\n8\n' | paste -sd+)"
echo "$conta = $((conta))"     # o Bash faz a conta com $(( ))
rm -r treino

## 🐍 Ponte Python

O mesmo resultado só com a biblioteca padrão do Python:

In [ ]:
!cd "$L4O_HOME/dias/dia-10-utilitarios-de-texto" && python3 equivalente.py

## 🧩 Desafio

Encontrou a flag? Troque o texto abaixo e rode a célula.

Dicas e passo a passo estão no README do dia.

In [ ]:
flag = 'VAZANTE{troque_aqui}'  # ✏️ cole sua resposta
!cd "$L4O_HOME/dias/dia-10-utilitarios-de-texto" && bash checar.sh "{flag}"

---

📝 Anote o que aprendeu no `DIARIO.md` do repositório.

Feito por [Deivison Lourenço](https://github.com/Ridd1kulusC0d3r) · curso original de [cipher387 / Cyber Detective](https://cybdetective.com)